In [1]:
import json
import time
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests


class PncpMergedFilteredClient:
    BASE_URL_CONSULTA = "https://pncp.gov.br/api/consulta"
    BASE_URL_SEARCH = "https://pncp.gov.br/api/search/"
    BASE_SITE_URL = "https://pncp.gov.br"

    def __init__(
        self,
        timeout: int = 60,
        sleep_seconds: float = 0.2,
        max_retries: int = 3,
        retry_wait: float = 1.5,
        session: Optional[requests.Session] = None,
    ) -> None:
        self.timeout = timeout
        self.sleep_seconds = sleep_seconds
        self.max_retries = max_retries
        self.retry_wait = retry_wait
        self.session = session or requests.Session()
        self.session.headers.update(
            {
                "User-Agent": "Mozilla/5.0",
                "Accept": "application/json",
            }
        )

    # =========================================================
    # Utilidades
    # =========================================================
    def _request_json(self, url: str, params: Dict[str, Any]) -> Any:
        last_error = None

        for tentativa in range(1, self.max_retries + 1):
            try:
                response = self.session.get(
                    url,
                    params=params,
                    timeout=self.timeout,
                )

                print(f"[DEBUG URL] {response.url}")

                if response.status_code == 400:
                    return {"_http_400": True}

                response.raise_for_status()
                return response.json()

            except requests.RequestException as e:
                last_error = e
                print(f"[WARN] Tentativa {tentativa}/{self.max_retries} falhou: {e}")

                if tentativa < self.max_retries:
                    time.sleep(self.retry_wait)

        raise last_error

    def _normalize_text(self, text: Optional[str]) -> str:
        if not text:
            return ""
        text = str(text).lower().strip()
        text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
        return " ".join(text.split())

    def _build_site_url(self, item_url: Optional[str]) -> Optional[str]:
        if not item_url:
            return None
        if item_url.startswith("http://") or item_url.startswith("https://"):
            return item_url
        return f"{self.BASE_SITE_URL}{item_url}"

    def _extract_numero_controle(self, item: Dict[str, Any]) -> Optional[str]:
        return (
            item.get("numeroControlePNCP")
            or item.get("numero_controle_pncp")
            or item.get("raw", {}).get("numero_controle_pncp")
            or item.get("raw", {}).get("numeroControlePNCP")
        )

    def _extract_data_atualizacao(self, item: Dict[str, Any]) -> str:
        return (
            item.get("dataAtualizacaoGlobal")
            or item.get("dataAtualizacaoPncp")
            or item.get("dataAtualizacao")
            or item.get("data_atualizacao_pncp")
            or item.get("raw", {}).get("data_atualizacao_pncp")
            or ""
        )

    def _extract_objeto(self, item: Dict[str, Any]) -> str:
        return (
            item.get("objetoCompra")
            or item.get("description")
            or item.get("raw", {}).get("description")
            or ""
        )

    # =========================================================
    # Normalização
    # =========================================================
    def _normalize_search_item(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "origem": "search",
            "objetoCompra": item.get("description"),
            "titulo": item.get("title"),
            "id": item.get("id"),
            "numeroControlePNCP": item.get("numero_controle_pncp"),
            "anoCompra": item.get("ano"),
            "sequencialCompra": item.get("numero_sequencial"),
            "numeroCompra": item.get("numero"),
            "processo": None,
            "documentType": item.get("document_type"),
            "tipoId": item.get("tipo_id"),
            "tipoNome": item.get("tipo_nome"),
            "modalidadeId": item.get("modalidade_licitacao_id"),
            "modalidadeNome": item.get("modalidade_licitacao_nome"),
            "situacaoCompraId": item.get("situacao_id"),
            "situacaoCompraNome": item.get("situacao_nome"),
            "orgaoEntidade": {
                "cnpj": item.get("orgao_cnpj"),
                "razaoSocial": item.get("orgao_nome"),
                "poderId": item.get("poder_id"),
                "esferaId": item.get("esfera_id"),
            },
            "unidadeOrgao": {
                "codigoUnidade": item.get("unidade_codigo"),
                "nomeUnidade": item.get("unidade_nome"),
                "municipioNome": item.get("municipio_nome"),
                "ufSigla": item.get("uf"),
            },
            "dataPublicacaoPncp": item.get("data_publicacao_pncp"),
            "dataAtualizacaoPncp": item.get("data_atualizacao_pncp"),
            "dataInicioVigencia": item.get("data_inicio_vigencia"),
            "dataFimVigencia": item.get("data_fim_vigencia"),
            "dataAssinatura": item.get("data_assinatura"),
            "cancelado": item.get("cancelado"),
            "temResultado": item.get("tem_resultado"),
            "valorTotalEstimado": item.get("valor_global"),
            "valorTotalHomologado": None,
            "itemUrl": item.get("item_url"),
            "urlPncp": self._build_site_url(item.get("item_url")),
            "raw": item,
        }

    def _normalize_consulta_item(self, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "origem": "consulta",
            "objetoCompra": item.get("objetoCompra"),
            "titulo": None,
            "id": item.get("numeroControlePNCP"),
            "numeroControlePNCP": item.get("numeroControlePNCP"),
            "anoCompra": item.get("anoCompra"),
            "sequencialCompra": item.get("sequencialCompra"),
            "numeroCompra": item.get("numeroCompra"),
            "processo": item.get("processo"),
            "documentType": "edital",
            "tipoId": item.get("tipoInstrumentoConvocatorioCodigo"),
            "tipoNome": item.get("tipoInstrumentoConvocatorioNome"),
            "modalidadeId": item.get("modalidadeId"),
            "modalidadeNome": item.get("modalidadeNome"),
            "situacaoCompraId": item.get("situacaoCompraId"),
            "situacaoCompraNome": item.get("situacaoCompraNome"),
            "orgaoEntidade": item.get("orgaoEntidade"),
            "unidadeOrgao": item.get("unidadeOrgao"),
            "dataPublicacaoPncp": item.get("dataPublicacaoPncp"),
            "dataAtualizacaoPncp": item.get("dataAtualizacaoGlobal") or item.get("dataAtualizacao"),
            "dataInicioVigencia": item.get("dataAberturaProposta"),
            "dataFimVigencia": item.get("dataEncerramentoProposta"),
            "dataAssinatura": None,
            "cancelado": False,
            "temResultado": item.get("valorTotalHomologado") is not None,
            "valorTotalEstimado": item.get("valorTotalEstimado"),
            "valorTotalHomologado": item.get("valorTotalHomologado"),
            "itemUrl": None,
            "urlPncp": None,
            "raw": item,
        }

    # =========================================================
    # Coleta endpoint search
    # =========================================================
    def fetch_search_items(
        self,
        tipos_documento: Optional[List[str]] = None,
        status: Optional[str] = None,
        ordenacao: str = "-data",
        tam_pagina: int = 100,
        max_paginas: Optional[int] = None,
        verbose: bool = True,
    ) -> List[Dict[str, Any]]:
        pagina = 1
        resultados: List[Dict[str, Any]] = []

        while True:
            if max_paginas is not None and pagina > max_paginas:
                if verbose:
                    print(f"[INFO] Limite de páginas do search atingido: {max_paginas}")
                break

            params: Dict[str, Any] = {
                "pagina": pagina,
                "tam_pagina": tam_pagina,
                "ordenacao": ordenacao,
            }

            if status:
                params["status"] = status

            if tipos_documento:
                params["tipos_documento"] = (
                    tipos_documento[0] if len(tipos_documento) == 1 else ",".join(tipos_documento)
                )

            data = self._request_json(self.BASE_URL_SEARCH, params)

            if data.get("_http_400"):
                if verbose:
                    print(f"[INFO] Search retornou 400 na página {pagina}. Encerrando.")
                break

            items = data.get("items", [])
            if verbose:
                print(f"[INFO] Search página {pagina}: {len(items)} itens")

            if not items:
                break

            resultados.extend([self._normalize_search_item(x) for x in items])

            if len(items) < tam_pagina:
                break

            pagina += 1
            if self.sleep_seconds > 0:
                time.sleep(self.sleep_seconds)

        return resultados

    # =========================================================
    # Coleta endpoint consulta/proposta
    # =========================================================
    def fetch_consulta_items(
        self,
        data_inicial: int,
        data_final: int,
        codigo_modalidades: Optional[List[int]] = None,
        tamanho_pagina: int = 50,
        max_paginas_por_modalidade: Optional[int] = None,
        verbose: bool = True,
    ) -> List[Dict[str, Any]]:
        if not codigo_modalidades:
            codigo_modalidades = [4, 5, 6, 7, 8, 10, 12]

        resultados: List[Dict[str, Any]] = []

        for modalidade in codigo_modalidades:
            pagina = 1

            while True:
                if (
                    max_paginas_por_modalidade is not None
                    and pagina > max_paginas_por_modalidade
                ):
                    if verbose:
                        print(
                            f"[INFO] Limite de páginas da modalidade {modalidade} atingido: "
                            f"{max_paginas_por_modalidade}"
                        )
                    break

                params = {
                    "dataInicial": data_inicial,
                    "dataFinal": data_final,
                    "codigoModalidadeContratacao": modalidade,
                    "pagina": pagina,
                    "tamanhoPagina": tamanho_pagina,
                }

                url = f"{self.BASE_URL_CONSULTA}/v1/contratacoes/proposta"
                data = self._request_json(url, params)

                if data.get("_http_400"):
                    if verbose:
                        print(
                            f"[INFO] Consulta/proposta retornou 400 na modalidade {modalidade}, "
                            f"página {pagina}. Encerrando modalidade."
                        )
                    break

                items = data.get("data", [])
                if verbose:
                    print(
                        f"[INFO] Consulta modalidade {modalidade} página {pagina}: {len(items)} itens"
                    )

                if not items:
                    break

                resultados.extend([self._normalize_consulta_item(x) for x in items])

                if len(items) < tamanho_pagina:
                    break

                pagina += 1
                if self.sleep_seconds > 0:
                    time.sleep(self.sleep_seconds)

        return resultados

    # =========================================================
    # Merge + filtro
    # =========================================================
    def merge_unique(
        self,
        items_1: List[Dict[str, Any]],
        items_2: List[Dict[str, Any]],
    ) -> List[Dict[str, Any]]:
        unicos: Dict[str, Dict[str, Any]] = {}
        sem_chave: List[Dict[str, Any]] = []

        for item in items_1 + items_2:
            chave = self._extract_numero_controle(item)

            if not chave:
                sem_chave.append(item)
                continue

            if chave not in unicos:
                unicos[chave] = item
            else:
                atual = unicos[chave]
                if self._extract_data_atualizacao(item) > self._extract_data_atualizacao(atual):
                    unicos[chave] = item

        return list(unicos.values()) + sem_chave

    def filter_by_keywords(
        self,
        items: List[Dict[str, Any]],
        keywords_any: Optional[List[str]] = None,
        extras: Optional[List[str]] = None,
    ) -> List[Dict[str, Any]]:
        keywords_any = keywords_any or []
        extras = extras or []

        termos = keywords_any + extras
        termos_norm = [self._normalize_text(t) for t in termos if t]

        if not termos_norm:
            return items

        filtrados: List[Dict[str, Any]] = []

        for item in items:
            objeto = self._normalize_text(self._extract_objeto(item))
            if any(termo in objeto for termo in termos_norm):
                filtrados.append(item)

        return filtrados

    # =========================================================
    # Exportação
    # =========================================================
    def salvar_json(self, dados: Any, caminho_arquivo: str) -> None:
        with open(caminho_arquivo, "w", encoding="utf-8") as f:
            json.dump(dados, f, ensure_ascii=False, indent=2)

    def salvar_csv(self, items: List[Dict[str, Any]], caminho_arquivo: str) -> None:
        df = pd.json_normalize(items)
        df.to_csv(caminho_arquivo, index=False, encoding="utf-8-sig")

    # =========================================================
    # Orquestração final
    # =========================================================
    def executar(
        self,
        data_inicial: int,
        data_final: int,
        caminho_json_total: str = "pncp_total_unicos.json",
        caminho_json_filtrado: str = "pncp_filtrados.json",
        caminho_csv_total: Optional[str] = None,
        caminho_csv_filtrado: Optional[str] = None,
        keywords_any: Optional[List[str]] = None,
        extras: Optional[List[str]] = None,
        tipos_documento_search: Optional[List[str]] = None,
        status_search: str = "recebendo_proposta",
        codigo_modalidades_consulta: Optional[List[int]] = None,
        max_paginas_search: Optional[int] = None,
        max_paginas_por_modalidade_consulta: Optional[int] = None,
        verbose: bool = True,
    ) -> Dict[str, Any]:
        itens_search = self.fetch_search_items(
            tipos_documento=tipos_documento_search or ["edital"],
            status=status_search,
            max_paginas=max_paginas_search,
            verbose=verbose,
        )

        itens_consulta = self.fetch_consulta_items(
            data_inicial=data_inicial,
            data_final=data_final,
            codigo_modalidades=codigo_modalidades_consulta,
            max_paginas_por_modalidade=max_paginas_por_modalidade_consulta,
            verbose=verbose,
        )

        itens_unicos = self.merge_unique(itens_search, itens_consulta)
        itens_filtrados = self.filter_by_keywords(
            itens_unicos,
            keywords_any=keywords_any,
            extras=extras,
        )

        payload_total = {
            "total_search": len(itens_search),
            "total_consulta": len(itens_consulta),
            "total_unicos": len(itens_unicos),
            "items": itens_unicos,
        }

        payload_filtrado = {
            "total_unicos_antes_do_filtro": len(itens_unicos),
            "total_filtrados": len(itens_filtrados),
            "keywords_any": keywords_any or [],
            "extras": extras or [],
            "termos_usados": (keywords_any or []) + (extras or []),
            "items": itens_filtrados,
        }

        self.salvar_json(payload_total, caminho_json_total)
        self.salvar_json(payload_filtrado, caminho_json_filtrado)

        if caminho_csv_total:
            self.salvar_csv(itens_unicos, caminho_csv_total)

        if caminho_csv_filtrado:
            self.salvar_csv(itens_filtrados, caminho_csv_filtrado)

        if verbose:
            print("\n[OK] Execução finalizada")
            print(f"JSON total: {caminho_json_total}")
            print(f"JSON filtrado: {caminho_json_filtrado}")
            print(f"Total search: {len(itens_search)}")
            print(f"Total consulta: {len(itens_consulta)}")
            print(f"Total únicos: {len(itens_unicos)}")
            print(f"Total filtrados: {len(itens_filtrados)}")

        return {
            "total": payload_total,
            "filtrado": payload_filtrado,
        }

In [2]:
keywords_any = [
    "Software de segurança",
    "Software de monitoramento",
    "Software para controle de portaria",
    "Software para registro de frequência",
    "software de gestão",
    "Relógio de ponto",
    "Cartão de ponto",
    "controle de ponto",
    "reconhecimento facial",
    "registrador eletrônico de ponto biométrico",
    "Gerenciamento Eletrônico de Frequência",
    "controle de frequência",
    "Registrador de ponto facial",
    "Sistema com aplicativo facial",
    "controle de frequência facial",
    "Controle de Faces",
    "Equipamento com biometria de face",
    "Identificador Facial",
    "bastão de ronda",
    "relógio protocolador",
    "relógio cartográfico",
    "Bobina Térmica para Relógio de Ponto",
    "Papel Termosensível para Relógio de Ponto",
    "CONTROLADOR DE ACESSO",
    "Fechadura Facial",
    "fechadura biométrica",
    "Fechadura Eletrônica Facial",
    "Fechadura eletromagnética",
    "Controladora de acesso facial",
    "HID",
    "Hickvision",
    "detector de metais",
    "porta giratória",
    "Portal Detector Metal",
    "porta automática",
    "motor de portão",
    "catracas faciais",
    "catraca balcão",
    "catraca mecânica",
    "catracas eletrônicas",
    "Catraca pedestal bidirecional",
    "catraca PNE",
    "controle de acesso",
    "controle de acesso facial",
    "cancela automática",
    "cancela eletrônica",
    "leitores faciais",
    "Torniquete",
    "Conjunto Controle Acesso Área Restrita",
    "iDFace",
    "botoeira",
]

extras = [
    "biometria",
    "facial",
    "portaria",
]

client = PncpMergedFilteredClient(
    timeout=60,
    sleep_seconds=0.2,
    max_retries=3,
    retry_wait=2,
)

resultado = client.executar(
    data_inicial=20260301,
    data_final=20260319,
    caminho_json_total="/Users/jose-cleiton/dev/script_pncp/pncp_total_unicos.json",
    caminho_json_filtrado="/Users/jose-cleiton/dev/script_pncp/pncp_filtrados.json",
    caminho_csv_total="/Users/jose-cleiton/dev/script_pncp/pncp_total_unicos.csv",
    caminho_csv_filtrado="/Users/jose-cleiton/dev/script_pncp/pncp_filtrados.csv",
    keywords_any=keywords_any,
    extras=extras,
    tipos_documento_search=["edital"],
    status_search="recebendo_proposta",
    codigo_modalidades_consulta=[4, 5, 6, 7, 8, 10, 12],
    max_paginas_search=None,
    max_paginas_por_modalidade_consulta=None,
    verbose=True,
)

[DEBUG URL] https://pncp.gov.br/api/search/?pagina=1&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 1: 100 itens
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=2&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 2: 100 itens
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=3&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 3: 100 itens
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=4&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 4: 100 itens
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=5&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 5: 100 itens
[DEBUG URL] https://pncp.gov.br/api/search/?pagina=6&tam_pagina=100&ordenacao=-data&status=recebendo_proposta&tipos_documento=edital
[INFO] Search página 6: 100 iten